In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
!pip install gpboost
import gpboost as gpb
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, roc_auc_score
)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


RADAR Analysis

Gradient Boosting Regressor 

- for prediction of continuous phq8 variable
- phq8 then transformed into categorical data to calculate metrics of the regressor

In [3]:
# -----------------------------
# Load and prepare data
# -----------------------------
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv")
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(dataset)

feature_cols = df.iloc[:, 12:-1].columns.tolist()
df = df.dropna(subset=feature_cols + ["phq8_score", "participant_id"]).copy()

# Binary label for classification-style metrics
df["depressed"] = (df["phq8_score"] >= 10).astype(int)

X = df[feature_cols].values
y = df["phq8_score"].values
y_bin = df["depressed"].values
groups = df["participant_id"].values

# -----------------------------
# Train / test split by participant
# -----------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_full, X_test = X[train_idx], X[test_idx]
y_train_full, y_test = y[train_idx], y[test_idx]
y_bin_train_full, y_bin_test = y_bin[train_idx], y_bin[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train participants:", len(np.unique(groups_train)))
print("Test participants:", len(np.unique(groups_test)))

# -----------------------------
# Cross-validation on training set only
# -----------------------------
gkf = GroupKFold(n_splits=5)

cv_results = []

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    gkf.split(X_train_full, y_train_full, groups=groups_train), start=1
):
    X_train, X_val = X_train_full[cv_train_idx], X_train_full[cv_val_idx]
    y_train, y_val = y_train_full[cv_train_idx], y_train_full[cv_val_idx]
    y_bin_val = y_bin_train_full[cv_val_idx]

    model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    preds_bin = (preds >= 10).astype(int)

    cv_results.append({
        "fold": fold,
        "mae": mean_absolute_error(y_val, preds),
        "rmse": mean_squared_error(y_val, preds) ** 0.5,
        "r2": r2_score(y_val, preds),
        "accuracy": accuracy_score(y_bin_val, preds_bin),
        "f1": f1_score(y_bin_val, preds_bin),
        "roc_auc": roc_auc_score(y_bin_val, preds)
    })

cv_results_df = pd.DataFrame(cv_results)

print("\nCV fold results:")
print(cv_results_df)

cv_summary_df = pd.DataFrame([{
    "subset": "cv_train",
    "n_rows": len(train_idx),
    "n_depressed": int(y_bin_train_full.sum()),
    "n_control": int((y_bin_train_full == 0).sum()),
    "mae_mean": cv_results_df["mae"].mean(),
    "mae_std": cv_results_df["mae"].std(),
    "rmse_mean": cv_results_df["rmse"].mean(),
    "rmse_std": cv_results_df["rmse"].std(),
    "r2_mean": cv_results_df["r2"].mean(),
    "r2_std": cv_results_df["r2"].std(),
    "accuracy_mean": cv_results_df["accuracy"].mean(),
    "accuracy_std": cv_results_df["accuracy"].std(),
    "f1_mean": cv_results_df["f1"].mean(),
    "f1_std": cv_results_df["f1"].std(),
    "roc_auc_mean": cv_results_df["roc_auc"].mean(),
    "roc_auc_std": cv_results_df["roc_auc"].std(),
}])

print("\nCV summary:")
print(cv_summary_df)

# -----------------------------
# Final model: train on all training data, evaluate on held-out test set
# -----------------------------
final_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

final_model.fit(X_train_full, y_train_full)
test_preds = final_model.predict(X_test)
test_preds_bin = (test_preds >= 10).astype(int)

test_summary_df = pd.DataFrame([{
    "subset": "held_out_test",
    "n_rows": len(test_idx),
    "n_depressed": int(y_bin_test.sum()),
    "n_control": int((y_bin_test == 0).sum()),
    "mae": mean_absolute_error(y_test, test_preds),
    "rmse": mean_squared_error(y_test, test_preds) ** 0.5,
    "r2": r2_score(y_test, test_preds),
    "accuracy": accuracy_score(y_bin_test, test_preds_bin),
    "f1": f1_score(y_bin_test, test_preds_bin),
    "roc_auc": roc_auc_score(y_bin_test, test_preds)
}])

print("\nHeld-out test results:")
print(test_summary_df)

# -----------------------------
# Save results
# -----------------------------
cv_results_df.to_csv(RESULTS_PATH / "Radar_xgb_cv_folds.csv", index=False)
cv_summary_df.to_csv(RESULTS_PATH / "Radar_xgb_cv_summary.csv", index=False)
test_summary_df.to_csv(RESULTS_PATH / "Radar_xgb_test_summary.csv", index=False)

Train rows: 6570
Test rows: 1945
Train participants: 219
Test participants: 55

CV fold results:
   fold       mae      rmse        r2  accuracy        f1   roc_auc
0     1  5.503637  6.693088  0.022327  0.571537  0.468366  0.620677
1     2  5.230133  6.099182 -0.102878  0.575342  0.401288  0.532761
2     3  5.410644  6.474827 -0.062007  0.527397  0.421249  0.534426
3     4  4.654820  5.740867 -0.022166  0.560122  0.496516  0.581427
4     5  4.360752  5.434725 -0.056557  0.611111  0.393832  0.583061

CV summary:
     subset  n_rows  n_depressed  n_control  mae_mean   mae_std  rmse_mean  \
0  cv_train    6570         2945       3625  5.031997  0.499477   6.088538   

   rmse_std   r2_mean    r2_std  accuracy_mean  accuracy_std  f1_mean  \
0  0.515548 -0.044256  0.046969       0.569102      0.030123  0.43625   

    f1_std  roc_auc_mean  roc_auc_std  
0  0.04447       0.57047      0.03715  

Held-out test results:
          subset  n_rows  n_depressed  n_control       mae      rmse  \
0 

Gradient Boosting Classifier

- phq8 transformed into categorical data (1,0) for binary classification
- multiclass classification?

- Gridsearch CV used to assess the best params for tuning

In [4]:
# -----------------------------
# Load and prepare data
# -----------------------------
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv")
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(dataset)

feature_cols = df.iloc[:, 12:-1].columns.tolist()
df = df.dropna(subset=feature_cols + ["phq8_score", "participant_id"]).copy()

# Binary target
df["depressed"] = (df["phq8_score"] >= 10).astype(int)

X = df[feature_cols].values
y_bin = df["depressed"].values
groups = df["participant_id"].values

# -----------------------------
# Held-out test split by participant
# -----------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y_bin, groups=groups))

X_train_full, X_test = X[train_idx], X[test_idx]
y_train_full, y_test = y_bin[train_idx], y_bin[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train participants:", len(np.unique(groups_train)))
print("Test participants:", len(np.unique(groups_test)))

# -----------------------------
# Cross-validation on training set only
# -----------------------------
gkf = GroupKFold(n_splits=5)
cv_results = []

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    gkf.split(X_train_full, y_train_full, groups=groups_train), start=1
):
    X_train, X_val = X_train_full[cv_train_idx], X_train_full[cv_val_idx]
    y_train, y_val = y_train_full[cv_train_idx], y_train_full[cv_val_idx]

    model = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    proba = model.predict_proba(X_val)[:, 1]

    cv_results.append({
        "fold": fold,
        "accuracy": accuracy_score(y_val, preds),
        "f1": f1_score(y_val, preds),
        "roc_auc": roc_auc_score(y_val, proba)
    })

cv_results_df = pd.DataFrame(cv_results)

print("\nCV fold results:")
print(cv_results_df)

cv_summary_df = pd.DataFrame([{
    "subset": "cv_train",
    "n_rows": len(train_idx),
    "n_depressed": int(y_train_full.sum()),
    "n_control": int((y_train_full == 0).sum()),
    "accuracy_mean": cv_results_df["accuracy"].mean(),
    "accuracy_std": cv_results_df["accuracy"].std(),
    "f1_mean": cv_results_df["f1"].mean(),
    "f1_std": cv_results_df["f1"].std(),
    "roc_auc_mean": cv_results_df["roc_auc"].mean(),
    "roc_auc_std": cv_results_df["roc_auc"].std(),
}])

print("\nCV summary:")
print(cv_summary_df)

# -----------------------------
# Final model on full training set
# -----------------------------
final_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

final_model.fit(X_train_full, y_train_full)

test_preds = final_model.predict(X_test)
test_proba = final_model.predict_proba(X_test)[:, 1]

test_summary_df = pd.DataFrame([{
    "subset": "held_out_test",
    "n_rows": len(test_idx),
    "n_depressed": int(y_test.sum()),
    "n_control": int((y_test == 0).sum()),
    "accuracy": accuracy_score(y_test, test_preds),
    "f1": f1_score(y_test, test_preds),
    "roc_auc": roc_auc_score(y_test, test_proba)
}])

print("\nHeld-out test results:")
print(test_summary_df)

# -----------------------------
# Save results
# -----------------------------
cv_results_df.to_csv(RESULTS_PATH / "radar_xgbc_cv_folds.csv", index=False)
cv_summary_df.to_csv(RESULTS_PATH / "radar_xgbc_cv_summary.csv", index=False)
test_summary_df.to_csv(RESULTS_PATH / "radar_xgbc_test_summary.csv", index=False)

Train rows: 6570
Test rows: 1945
Train participants: 219
Test participants: 55

CV fold results:
   fold  accuracy        f1   roc_auc
0     1  0.576104  0.494096  0.610421
1     2  0.572298  0.382418  0.525857
2     3  0.538813  0.423954  0.521741
3     4  0.533486  0.425492  0.572962
4     5  0.592846  0.402235  0.563187

CV summary:
     subset  n_rows  n_depressed  n_control  accuracy_mean  accuracy_std  \
0  cv_train    6570         2945       3625       0.562709      0.025518   

    f1_mean    f1_std  roc_auc_mean  roc_auc_std  
0  0.425639  0.042146      0.558834     0.036548  

Held-out test results:
          subset  n_rows  n_depressed  n_control  accuracy        f1   roc_auc
0  held_out_test    1945          667       1278  0.580463  0.385542  0.545464


GPBoost accounts for group random effects by selecting a source of data clustering.

In this case, ID of the participants served as the source of clustering.

In [8]:

# -----------------------------
# Paths
# -----------------------------
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv")
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load and prepare data
# -----------------------------
df = pd.read_csv(dataset)

feature_cols = df.iloc[:, 12:-1].columns.tolist()

df = df.dropna(subset=feature_cols + ["phq8_score", "participant_id"]).copy()
df["depressed"] = (df["phq8_score"] >= 10).astype(int)
df["group_code"] = pd.factorize(df["participant_id"])[0].astype(np.int32)

X = df[feature_cols].values
y = df["depressed"].astype(int).values
groups = df["group_code"].values

# -----------------------------
# Held-out test split by participant
# -----------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_full, X_test = X[train_idx], X[test_idx]
y_train_full, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train participants:", len(np.unique(groups_train)))
print("Test participants:", len(np.unique(groups_test)))

# -----------------------------
# Cross-validation on training set only
# -----------------------------
gkf = GroupKFold(n_splits=5)
fold_results = []

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    gkf.split(X_train_full, y_train_full, groups=groups_train), start=1
):
    X_train, X_val = X_train_full[cv_train_idx], X_train_full[cv_val_idx]
    y_train, y_val = y_train_full[cv_train_idx], y_train_full[cv_val_idx]
    group_train = groups_train[cv_train_idx].astype(np.int32)
    group_val = groups_train[cv_val_idx].astype(np.int32)

    gp_model = gpb.GPModel(group_data=group_train)
    train_data = gpb.Dataset(X_train, label=y_train)

    params = {
        "objective": "binary",
        "learning_rate": 0.05,
        "max_depth": 4,
        "verbose": 0
    }

    gpbst = gpb.train(
        params=params,
        train_set=train_data,
        gp_model=gp_model,
        num_boost_round=200
    )

    pred = gpbst.predict(
        data=X_val,
        group_data_pred=group_val,
        predict_var=False
    )

    p_gpboost = pred["response_mean"]
    y_pred = (p_gpboost >= 0.5).astype(int)

    fold_results.append({
        "fold": fold,
        "accuracy": accuracy_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred),
        "roc_auc": roc_auc_score(y_val, p_gpboost)
    })

# Fold-level CV results
fold_results_df = pd.DataFrame(fold_results)
print("CV fold-level results:")
print(fold_results_df)

# CV summary
cv_summary_df = pd.DataFrame([{
    "subset": "cv_train",
    "n_rows": len(train_idx),
    "n_depressed": int(y_train_full.sum()),
    "n_control": int((y_train_full == 0).sum()),
    "accuracy_mean": fold_results_df["accuracy"].mean(),
    "accuracy_std": fold_results_df["accuracy"].std(),
    "f1_mean": fold_results_df["f1"].mean(),
    "f1_std": fold_results_df["f1"].std(),
    "roc_auc_mean": fold_results_df["roc_auc"].mean(),
    "roc_auc_std": fold_results_df["roc_auc"].std()
}])

print("\nCV summary:")
print(cv_summary_df)

# -----------------------------
# Final GPBoost model on full training set
# -----------------------------
gp_model_final = gpb.GPModel(group_data=groups_train.astype(np.int32))
train_data_final = gpb.Dataset(X_train_full, label=y_train_full)

params = {
    "objective": "binary",
    "learning_rate": 0.05,
    "max_depth": 4,
    "verbose": 0
}

gpbst_final = gpb.train(
    params=params,
    train_set=train_data_final,
    gp_model=gp_model_final,
    num_boost_round=200
)

test_pred = gpbst_final.predict(
    data=X_test,
    group_data_pred=groups_test.astype(np.int32),
    predict_var=False
)

test_proba = test_pred["response_mean"]
test_preds = (test_proba >= 0.5).astype(int)

test_summary_df = pd.DataFrame([{
    "subset": "held_out_test",
    "n_rows": len(test_idx),
    "n_depressed": int(y_test.sum()),
    "n_control": int((y_test == 0).sum()),
    "accuracy": accuracy_score(y_test, test_preds),
    "f1": f1_score(y_test, test_preds),
    "roc_auc": roc_auc_score(y_test, test_proba)
}])

print("\nHeld-out test results:")
print(test_summary_df)

# -----------------------------
# Save results
# -----------------------------
fold_results_df.to_csv(RESULTS_PATH / "radar_gpboost_cv_folds.csv", index=False)
cv_summary_df.to_csv(RESULTS_PATH / "radar_gpboost_cv_summary.csv", index=False)
test_summary_df.to_csv(RESULTS_PATH / "radar_gpboost_test_summary.csv", index=False)

Train rows: 6570
Test rows: 1945
Train participants: 219
Test participants: 55
[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the GPModel is changed accordingly. This can be problematic if the GPModel has been pre-trained 
[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the GPModel is changed accordingly. This can be problematic if the GPModel has been pre-trained 
[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the GPModel is changed accordingly. This can be problematic if the 

Androids Analysis

In [9]:
# Paths
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/androids_model_dataset_basic.csv")
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/GradBoost/Androids")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Load
df = pd.read_csv(dataset)

meta_cols = [
    "file_path", "file", "file_stem", "bdi_score",
    "depressed", "fold", "speech_type", "subgroup_from_path"
]

feature_cols = [c for c in df.columns if c not in meta_cols]

df = df.dropna(subset=feature_cols + ["depressed", "file_stem"]).copy()

X = df[feature_cols].values
y = df["depressed"].astype(int).values
groups = df["file_stem"].values

# Train/test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_full, X_test = X[train_idx], X[test_idx]
y_train_full, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

# CV on training set
gkf = GroupKFold(n_splits=5)
cv_results = []

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    gkf.split(X_train_full, y_train_full, groups=groups_train), start=1
):
    X_train, X_val = X_train_full[cv_train_idx], X_train_full[cv_val_idx]
    y_train, y_val = y_train_full[cv_train_idx], y_train_full[cv_val_idx]

    model = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    proba = model.predict_proba(X_val)[:, 1]

    cv_results.append({
        "fold": fold,
        "accuracy": accuracy_score(y_val, preds),
        "f1": f1_score(y_val, preds),
        "roc_auc": roc_auc_score(y_val, proba)
    })

cv_results_df = pd.DataFrame(cv_results)
print("CV fold results:")
print(cv_results_df)

cv_summary_df = pd.DataFrame([{
    "subset": "cv_train",
    "n_rows": len(train_idx),
    "n_depressed": int(y_train_full.sum()),
    "n_control": int((y_train_full == 0).sum()),
    "accuracy_mean": cv_results_df["accuracy"].mean(),
    "accuracy_std": cv_results_df["accuracy"].std(),
    "f1_mean": cv_results_df["f1"].mean(),
    "f1_std": cv_results_df["f1"].std(),
    "roc_auc_mean": cv_results_df["roc_auc"].mean(),
    "roc_auc_std": cv_results_df["roc_auc"].std(),
}])

print("\nCV summary:")
print(cv_summary_df)

# Final model on full training set
final_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

final_model.fit(X_train_full, y_train_full)
test_preds = final_model.predict(X_test)
test_proba = final_model.predict_proba(X_test)[:, 1]

test_summary_df = pd.DataFrame([{
    "subset": "held_out_test",
    "n_rows": len(test_idx),
    "n_depressed": int(y_test.sum()),
    "n_control": int((y_test == 0).sum()),
    "accuracy": accuracy_score(y_test, test_preds),
    "f1": f1_score(y_test, test_preds),
    "roc_auc": roc_auc_score(y_test, test_proba)
}])

print("\nHeld-out test results:")
print(test_summary_df)

cv_results_df.to_csv(RESULTS_PATH / "androids_gbc_cv_folds.csv", index=False)
cv_summary_df.to_csv(RESULTS_PATH / "androids_gbc_cv_summary.csv", index=False)
test_summary_df.to_csv(RESULTS_PATH / "androids_gbc_test_summary.csv", index=False)

CV fold results:
   fold  accuracy        f1   roc_auc
0     1  0.444444  0.500000  0.467532
1     2  0.666667  0.727273  0.725000
2     3  0.722222  0.705882  0.793750
3     4  0.638889  0.763636  0.492308
4     5  0.628571  0.682927  0.640523

CV summary:
     subset  n_rows  n_depressed  n_control  accuracy_mean  accuracy_std  \
0  cv_train     179          101         78       0.620159      0.104734   

    f1_mean    f1_std  roc_auc_mean  roc_auc_std  
0  0.675944  0.102745      0.623823     0.142402  

Held-out test results:
          subset  n_rows  n_depressed  n_control  accuracy        f1   roc_auc
0  held_out_test      45           21         24  0.733333  0.714286  0.746032


In [11]:
# Paths
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/androids_model_dataset_basic.csv")
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/GradBoost/Androids")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Load
df = pd.read_csv(dataset)

meta_cols = [
    "file_path", "file", "file_stem", "bdi_score",
    "depressed", "fold", "speech_type", "subgroup_from_path"
]

feature_cols = [c for c in df.columns if c not in meta_cols]

df = df.dropna(subset=feature_cols + ["depressed", "file_stem"]).copy()
df["group_code"] = pd.factorize(df["file_stem"])[0].astype(np.int32)

X = df[feature_cols].values
y = df["depressed"].astype(int).values
groups = df["group_code"].values

# Train/test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_full, X_test = X[train_idx], X[test_idx]
y_train_full, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

# CV on training set
gkf = GroupKFold(n_splits=5)
fold_results = []

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    gkf.split(X_train_full, y_train_full, groups=groups_train), start=1
):
    X_train, X_val = X_train_full[cv_train_idx], X_train_full[cv_val_idx]
    y_train, y_val = y_train_full[cv_train_idx], y_train_full[cv_val_idx]
    group_train = groups_train[cv_train_idx].astype(np.int32)
    group_val = groups_train[cv_val_idx].astype(np.int32)

    gp_model = gpb.GPModel(group_data=group_train)
    train_data = gpb.Dataset(X_train, label=y_train)

    params = {
        "objective": "binary",
        "learning_rate": 0.05,
        "max_depth": 4,
        "verbose": 0
    }

    gpbst = gpb.train(
        params=params,
        train_set=train_data,
        gp_model=gp_model,
        num_boost_round=200
    )

    pred = gpbst.predict(
        data=X_val,
        group_data_pred=group_val,
        predict_var=False
    )

    proba = pred["response_mean"]
    preds = (proba >= 0.5).astype(int)

    fold_results.append({
        "fold": fold,
        "accuracy": accuracy_score(y_val, preds),
        "f1": f1_score(y_val, preds),
        "roc_auc": roc_auc_score(y_val, proba)
    })

fold_results_df = pd.DataFrame(fold_results)
print("CV fold results:")
print(fold_results_df)

cv_summary_df = pd.DataFrame([{
    "subset": "cv_train",
    "n_rows": len(train_idx),
    "n_depressed": int(y_train_full.sum()),
    "n_control": int((y_train_full == 0).sum()),
    "accuracy_mean": fold_results_df["accuracy"].mean(),
    "accuracy_std": fold_results_df["accuracy"].std(),
    "f1_mean": fold_results_df["f1"].mean(),
    "f1_std": fold_results_df["f1"].std(),
    "roc_auc_mean": fold_results_df["roc_auc"].mean(),
    "roc_auc_std": fold_results_df["roc_auc"].std()
}])

print("\nCV summary:")
print(cv_summary_df)

# Final model on full training set
gp_model_final = gpb.GPModel(group_data=groups_train.astype(np.int32))
train_data_final = gpb.Dataset(X_train_full, label=y_train_full)

params = {
    "objective": "binary",
    "learning_rate": 0.05,
    "max_depth": 4,
    "verbose": 0
}

gpbst_final = gpb.train(
    params=params,
    train_set=train_data_final,
    gp_model=gp_model_final,
    num_boost_round=200
)

test_pred = gpbst_final.predict(
    data=X_test,
    group_data_pred=groups_test.astype(np.int32),
    predict_var=False
)

test_proba = test_pred["response_mean"]
test_preds = (test_proba >= 0.5).astype(int)

test_summary_df = pd.DataFrame([{
    "subset": "held_out_test",
    "n_rows": len(test_idx),
    "n_depressed": int(y_test.sum()),
    "n_control": int((y_test == 0).sum()),
    "accuracy": accuracy_score(y_test, test_preds),
    "f1": f1_score(y_test, test_preds),
    "roc_auc": roc_auc_score(y_test, test_proba)
}])

print("\nHeld-out test results:")
print(test_summary_df)

fold_results_df.to_csv(RESULTS_PATH / "androids_gpboost_cv_folds.csv", index=False)
cv_summary_df.to_csv(RESULTS_PATH / "androids_gpboost_cv_summary.csv", index=False)
test_summary_df.to_csv(RESULTS_PATH / "androids_gpboost_test_summary.csv", index=False)

[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the GPModel is changed accordingly. This can be problematic if the GPModel has been pre-trained 
[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the GPModel is changed accordingly. This can be problematic if the GPModel has been pre-trained 
[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the GPModel is changed accordingly. This can be problematic if the GPModel has been pre-trained 
[GPBoost] [Warning] The 'objective' (='binary') f